In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 영화 정보를 위한 Pydantic 모델 선언
from pydantic import BaseModel, Field

In [2]:
# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

`Field(..., description="영화 제목")`에서 `...` (Ellipsis, 생략 부호)는 **Pydantic** 라이브러리에서 **"이 필드는 필수(Required) 값이다"**라는 것을 명시하는 문법입니다.

구체적인 의미는 다음과 같습니다.

### 1. 필수 필드 지정 (Required Field)
Pydantic 모델을 생성할 때 해당 필드에 기본값을 주지 않고, 반드시 사용자가 값을 입력해야 한다는 뜻입니다.

*   **`...`을 사용하는 경우 (필수):**
    ```python
    title: str = Field(..., description="영화 제목")
    # 객체 생성 시 title을 안 넣으면 에러가 납니다.
    # movie = Movie() -> 에러 발생!
    ```

*   **기본값을 사용하는 경우 (선택):**
    ```python
    title: str = Field("제목 없음", description="영화 제목")
    # title을 안 넣으면 "제목 없음"이 자동으로 들어갑니다.
    # movie = Movie() -> title="제목 없음"으로 생성됨
    ```

### 2. 왜 `None` 대신 `...`을 쓰나요?
파이썬에서 `title: str = None`이라고 쓰면 "기본값이 None이다"라는 뜻이 되어버립니다. 하지만 **"기본값은 없지만, 설명(description) 같은 메타데이터는 추가하고 싶을 때"** `Field` 함수 안에서 첫 번째 인자로 `...`을 넣어 "이건 필수값이야!"라고 알려주는 것입니다.

### 3. LangChain/LLM에서의 역할
LangChain에서 LLM이 특정 구조로 답변을 생성하게 할 때(Structured Output), 이 `...`이 붙은 필드는 LLM에게 **"이 정보는 반드시 추출하거나 생성해야 해"**라고 강제하는 역할을 합니다.

### 요약
- `...` = **"이 값은 필수야, 꼭 넣어줘!"**
- 만약 `...` 자리에 다른 값을 넣으면 그 값이 **기본값**이 됩니다.

In [6]:
# Stream
for chunk in model.stream('Explain about the movie The Truman Show, Reply it briefly'):
  print(chunk.text, end='')

"The Truman Show," directed by Peter Weir and released in 1998, is a satirical drama that follows Truman Burbank, played by Jim Carrey, who unknowingly lives his entire life inside a massive reality television set. His every move is broadcast to the world, and every person in his life, including his friends and family, is an actor. As he begins to suspect that something is amiss, Truman's quest for truth and freedom leads him to confront the show's creator, Christof, portrayed by Ed Harris. The film explores themes of reality, free will, and the ethics of entertainment, prompting viewers to reflect on the nature of life and the impact of media.

In [10]:
# Batch
inputs = [
  'Explain about the movie The Truman Show',
  'Explain about the movie The Truman Show, Reply it briefly'
]

for response in model.batch(inputs):
  print(response.content)


"The Truman Show" is a 1998 American satirical science fiction psychological comedy-drama film directed by Peter Weir and written by Andrew Niccol. The film stars Jim Carrey as Truman Burbank, a man who unwittingly lives his entire life inside a massive television set, which is broadcast to the world as a reality show.

### Plot Overview
Truman Burbank is an ordinary man who works as an insurance salesman in the seemingly perfect, idyllic town of Seahaven. Unbeknownst to him, from the day he was born, his life has been the subject of a 24/7 reality television show, created and controlled by a man named Christof (played by Ed Harris), who acts as the show's producer. The residents of Seahaven are all actors, and Truman's life is scripted for entertainment purposes.

As the film progresses, Truman begins to notice inconsistencies in his environment that provoke his curiosity. He experiences strange occurrences that make him question the reality of his life. After he starts to suspect tha

In [12]:
class Movie(BaseModel):
  """영화 정보"""
  title: str = Field(..., description="영화 제목")
  director: str = Field(..., description="감독")
  year: int = Field(..., description="개봉 연도")
  genre: str = Field(..., description="장b르")

In [13]:
# Structured Output
# Movie 구조로 출력하도록 모델 래핑
mode_with_structured_output = model.with_structured_output(Movie)
response = mode_with_structured_output.invoke("Explain about the movie Truman Show")
print(response)
# title='The Truman Show' director='Peter Weir' year=1998 genre='Psychological comedy-drama'


title='The Truman Show' director='Peter Weir' year=1998 genre='Drama, Sci-Fi'


/Users/dohyunkim/Documents/langchain-for-ai-agent/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Movie(title='The Truman S..., genre='Drama, Sci-Fi'), input_type=Movie])
  return self.__pydantic_serializer__.to_python(


In [14]:
# 영화 정보를 위한 JSON Schema 선언
movie_json_schema = {
  "title": "Movie",
  "type": "object",
  "properties": {
    "title": {"type": "string", "description": "영화 제목"},
    "director": {"type": "string", "description": "감독"},
    "year": {"type": "integer", "description": "개봉 연도"},
    "genre": {"type": "string", "description": "장르"}
  },
  "required": ["title", "director", "year", "genre"]
}

model_with_json_schema = model.with_structured_output(movie_json_schema)
response = model_with_json_schema.invoke("Explain about the movie Truman Show")
print(response)
# title='The Truman Show' director='Peter Weir' year=1998 genre='Psychological comedy-drama'




{'title': 'The Truman Show', 'director': 'Peter Weir', 'year': 1998, 'genre': 'Drama, Sci-Fi, Comedy'}
